# Model selection without test leakage
**The Rental Price Puzzle · Predictive regression**  
Cyril James Lunzaga

Training-only property-grouped cross-validation selects a model before looking at held-out labels. This notebook reads the aggregated CV and evaluation scores produced by the frozen pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "portfolio" / "tables").is_dir():
    project_root = project_root.parent
root = project_root / "portfolio" / "tables"
assert root.is_dir(), "Open from the repository root or notebooks folder, and generate portfolio tables first"
cv = pd.read_csv(root / "grouped_cv_model_comparison.csv")
heldout = pd.read_csv(root / "heldout_model_comparison.csv")
assert set(cv["model"]) == {"median_baseline", "ridge_log_rent", "random_forest_log_rent"}
display(cv[["model", "mean_cv_mae_usd", "mean_cv_rmse_usd", "mean_cv_r2"]])

## The training-stage decision

Mean absolute error (USD/month) is the preselected ranking metric. The median benchmark predicts only the training median. Ridge and Random Forest use log-transformed targets with training-fold preprocessing. Ridge's numeric transform was stabilized during training, before held-out evaluation.

In [ ]:
selected = cv.sort_values("mean_cv_mae_usd").iloc[0]
print("Training-selected model:", selected["model"])
print(f"Mean grouped CV MAE: ${selected['mean_cv_mae_usd']:,.2f}")
assert selected["model"] == "random_forest_log_rent"
fig, ax = plt.subplots(figsize=(8, 4))
rows = cv.sort_values("mean_cv_mae_usd", ascending=False)
ax.barh(rows["model"].str.replace("_", " "), rows["mean_cv_mae_usd"])
ax.set(xlabel="Training CV MAE (USD per month; lower is better)", title="Selection by three-fold grouped CV")
fig.tight_layout()
plt.show()

## Primary holdout

The selected specification is refit on 79,196 listings and scored once against 19,905 group-separated listings. The held-out MAE and R² are independent of the training-stage model ranking.

In [ ]:
primary = heldout.loc[heldout["evaluation"].eq("primary")].set_index("model")
forest = primary.loc["random_forest_log_rent"]
baseline = primary.loc["median_baseline"]
improvement = 100 * (baseline["mae_usd"] - forest["mae_usd"]) / baseline["mae_usd"]
print(f"Held-out median baseline MAE: ${baseline['mae_usd']:,.2f}")
print(f"Held-out random forest MAE: ${forest['mae_usd']:,.2f}")
print(f"Reduction in mean absolute error: {improvement:.1f}%")
print(f"Held-out R²: {forest['r2']:.3f}")
assert forest["mae_usd"] < baseline["mae_usd"]
display(heldout)

**Interpretation.** The primary result supports a scope-limited retrospective rent-estimation finding. It is not a prospective lease-price or affordability prediction, and the geographic holdout must be read before discussing generalization.